# 🚀 GPU Multi-Style Trading Model Training
# Single Cell - Memory Optimized - Ultra Logged

**Run this cell and wait. Everything happens automatically.**

- 10 XGBoost GPU models
- Full metrics + progress bars
- Auto-download JSON report at end

In [ ]:
# ============================================================================
# COMPLETE TRAINING PIPELINE - SINGLE CELL
# ============================================================================

import json
import time
import warnings
import gc
import os
from datetime import datetime

import numpy as np
import polars as pl
import xgboost as xgb
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

print("="*80)
print("🔧 INITIALIZATION")
print("="*80)
print(f"[{datetime.now().strftime('%H:%M:%S')}] XGBoost: {xgb.__version__}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] Polars:  {pl.__version__}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] NumPy:   {np.__version__}")

try:
    gpu_available = "cuda" in xgb.build_info()["BUILD_FLAGS"]
    print(f"[{datetime.now().strftime('%H:%M:%S')}] GPU:     {gpu_available}")
except:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] GPU:     Unknown")

ram = psutil.virtual_memory()
print(f"[{datetime.now().strftime('%H:%M:%S')}] RAM:     {ram.total/(1024**3):.1f}GB total, {ram.available/(1024**3):.1f}GB free")
print("="*80)

# ============================================================================
# MOUNT GOOGLE DRIVE
# ============================================================================

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 📂 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ Drive mounted")

DATASET_PATH = "/content/drive/MyDrive/hydra_xauusd_feature_fabric_v1.parquet"

if os.path.exists(DATASET_PATH):
    size_gb = os.path.getsize(DATASET_PATH) / (1024**3)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ Dataset found: {size_gb:.2f}GB")
else:
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

# ============================================================================
# MODEL CONFIGURATIONS
# ============================================================================

MODEL_CONFIGS = [
    {"name": "XGB_Scalper_Fast", "horizon": 5, "style": "scalper", 
     "params": {"max_depth": 12, "eta": 0.05, "n_estimators": 500, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_Scalper_Deep", "horizon": 5, "style": "scalper", 
     "params": {"max_depth": 18, "eta": 0.03, "n_estimators": 600, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_Scalper_Aggressive", "horizon": 5, "style": "scalper", 
     "params": {"max_depth": 20, "eta": 0.02, "n_estimators": 700, "subsample": 0.95, "colsample_bytree": 0.95, "gamma": 0.05}},
    {"name": "XGB_DayTrader_Fast", "horizon": 72, "style": "day_trader", 
     "params": {"max_depth": 15, "eta": 0.05, "n_estimators": 500, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_DayTrader_Deep", "horizon": 72, "style": "day_trader", 
     "params": {"max_depth": 22, "eta": 0.03, "n_estimators": 700, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_DayTrader_Aggressive", "horizon": 72, "style": "day_trader", 
     "params": {"max_depth": 25, "eta": 0.02, "n_estimators": 800, "subsample": 0.95, "colsample_bytree": 0.95, "gamma": 0.05}},
    {"name": "XGB_SwingTrader_Fast", "horizon": 288, "style": "swing_trader", 
     "params": {"max_depth": 18, "eta": 0.05, "n_estimators": 500, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_SwingTrader_Deep", "horizon": 288, "style": "swing_trader", 
     "params": {"max_depth": 28, "eta": 0.03, "n_estimators": 800, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_MultiStyle_Light", "horizon": "multi", "style": "all", 
     "params": {"max_depth": 15, "eta": 0.05, "n_estimators": 600, "subsample": 0.9, "colsample_bytree": 0.9}},
    {"name": "XGB_MultiStyle_Deep", "horizon": "multi", "style": "all", 
     "params": {"max_depth": 25, "eta": 0.03, "n_estimators": 900, "subsample": 0.95, "colsample_bytree": 0.95, "gamma": 0.05}},
]

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 📋 Configured {len(MODEL_CONFIGS)} models")

# ============================================================================
# LOAD DATASET (MEMORY-EFFICIENT)
# ============================================================================

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] 📊 LOADING DATASET")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")

load_start = time.time()

print(f"[{datetime.now().strftime('%H:%M:%S')}] Step 1/5: Reading parquet...")
df = pl.read_parquet(DATASET_PATH)
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Loaded {df.shape[0]:,} rows × {df.shape[1]:,} cols")

print(f"[{datetime.now().strftime('%H:%M:%S')}] Step 2/5: Identifying features...")
exclude_cols = {"time", "open", "high", "low", "close", "tick_volume", "spread", "real_volume"}
label_cols = {f"label_{h}b" for h in [5, 10, 20, 72, 144, 288]}
fwd_ret_cols = {f"fwd_ret_{h}b" for h in [5, 10, 20, 72, 144, 288]}
exclude_cols.update(label_cols)
exclude_cols.update(fwd_ret_cols)
feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Features: {len(feature_cols)}")

print(f"[{datetime.now().strftime('%H:%M:%S')}] Step 3/5: Selecting columns...")
target_cols = [f"label_{h}b" for h in [5, 72, 288]]
df_select = df.select(["time"] + feature_cols + target_cols)
del df
gc.collect()
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Selected {df_select.shape[1]} columns")

print(f"[{datetime.now().strftime('%H:%M:%S')}] Step 4/5: Cleaning NaN...")
target_mask = pl.all_horizontal([pl.col(c).is_not_null() for c in target_cols])
df_clean = df_select.filter(target_mask)
del df_select
df_clean = df_clean.with_columns([pl.col(c).fill_null(0.0) for c in feature_cols])
print(f"[{datetime.now().strftime('%H:%M:%S')}]   After cleaning: {df_clean.shape[0]:,} rows")

print(f"[{datetime.now().strftime('%H:%M:%S')}] Step 5/5: Train/test split...")
split_idx = int(len(df_clean) * 0.7)

# Convert to numpy with reduced precision
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Converting to numpy (float16)...")
X = df_clean.select(feature_cols).to_numpy().astype(np.float16)
y = {h: df_clean[f"label_{h}b"].to_numpy().astype(np.int8) for h in [5, 72, 288]}
del df_clean
gc.collect()

print(f"[{datetime.now().strftime('%H:%M:%S')}]   Splitting...")
X_train, X_test = X[:split_idx], X[split_idx:]
del X
y_train = {h: y[h][:split_idx] for h in [5, 72, 288]}
y_test = {h: y[h][split_idx:] for h in [5, 72, 288]}
del y
gc.collect()

load_time = time.time() - load_start

print(f"[{datetime.now().strftime('%H:%M:%S')}]")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Train: {X_train.shape[0]:,} samples")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Test:  {X_test.shape[0]:,} samples")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Features: {X_train.shape[1]}")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Memory: Train={X_train.nbytes/(1024**3):.2f}GB, Test={X_test.nbytes/(1024**3):.2f}GB")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Load time: {load_time:.1f}s")

print(f"[{datetime.now().strftime('%H:%M:%S')}]")
print(f"[{datetime.now().strftime('%H:%M:%S')}] 📊 Label Distribution:")
for h in [5, 72, 288]:
    unique, counts = np.unique(y_train[h], return_counts=True)
    dist = {int(u): int(c) for u, c in zip(unique, counts)}
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   H{h:3d}: SELL={dist.get(-1,0):6,d} | HOLD={dist.get(0,0):6,d} | BUY={dist.get(1,0):6,d}")

print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")

data = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_cols": feature_cols,
}

gc.collect()

# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_model(config, data, model_idx, total_models):
    """Train single model with full logging."""
    
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] {'#'*80}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] MODEL {model_idx}/{total_models}: {config['name']}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'#'*80}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Style: {config['style']} | Horizon: {config['horizon']} bars")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Depth: {config['params']['max_depth']} | LR: {config['params']['eta']} | Trees: {config['params']['n_estimators']}")
    
    start_time = time.time()
    
    # Prepare targets (remap -1/0/1 -> 0/1/2)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Preparing targets...")
    if config["horizon"] == "multi":
        y_train = np.round(np.stack([data["y_train"][h] for h in [5, 72, 288]], axis=1).mean(axis=1)).astype(np.int8) + 1
        y_test = np.round(np.stack([data["y_test"][h] for h in [5, 72, 288]], axis=1).mean(axis=1)).astype(np.int8) + 1
    else:
        y_train = data["y_train"][config["horizon"]].astype(np.int8) + 1
        y_test = data["y_test"][config["horizon"]].astype(np.int8) + 1
    
    unique, counts = np.unique(y_train, return_counts=True)
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Class balance: SELL={counts[0]:,} | HOLD={counts[1]:,} | BUY={counts[2]:,}")
    
    # Class weights
    class_weights = len(y_train) / (len(unique) * counts)
    sample_weights = class_weights[y_train].astype(np.float32)
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Class weights: {class_weights}")
    
    # DMatrix
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Building DMatrix...")
    dtrain = xgb.DMatrix(data["X_train"].astype(np.float32), label=y_train, weight=sample_weights)
    dtest = xgb.DMatrix(data["X_test"].astype(np.float32), label=y_test)
    del sample_weights
    gc.collect()
    
    # Parameters
    params = config["params"].copy()
    n_est = params.pop("n_estimators", 500)
    params.update({
        "objective": "multi:softmax",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "gpu_hist",
        "device": "cuda",
        "verbosity": 0,
    })
    
    # Train
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Training {n_est} trees on GPU (early stop=50)...")
    train_losses = []
    test_losses = []
    
    with tqdm(total=n_est, desc=f"  {config['name'][:30]}", unit="tree", ncols=100, leave=True) as pbar:
        def callback(env):
            train_losses.append(env.evaluation_result_list[0][1])
            test_losses.append(env.evaluation_result_list[1][1])
            pbar.update(1)
            if (env.iteration + 1) % 50 == 0:
                pbar.set_postfix({"loss": f"{test_losses[-1]:.4f}", "iter": env.iteration + 1})
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=n_est,
            evals=[(dtrain, "train"), (dtest, "test")],
            early_stopping_rounds=50,
            verbose_eval=False,
            callbacks=[callback]
        )
    
    train_time = time.time() - start_time
    best_iter = model.best_iteration if hasattr(model, 'best_iteration') else n_est
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Best iteration: {best_iter}/{n_est}")
    
    # Predictions
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Predicting...")
    y_pred_train = model.predict(dtrain)
    y_pred_test = model.predict(dtest)
    
    # Metrics
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_prec = precision_score(y_test, y_pred_test, average="weighted", zero_division=0)
    test_rec = recall_score(y_test, y_pred_test, average="weighted", zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, average="weighted", zero_division=0)
    cm = confusion_matrix(y_test, y_pred_test)
    
    # Per-class accuracy
    per_class_acc = {}
    for cls_idx, cls_name in enumerate(["SELL", "HOLD", "BUY"]):
        mask = y_test == cls_idx
        if mask.sum() > 0:
            per_class_acc[cls_name] = accuracy_score(y_test[mask], y_pred_test[mask])
    
    # Feature importance
    top_10_feats = []
    try:
        feat_imp = model.get_score(importance_type="weight")
        top_10 = sorted(feat_imp.items(), key=lambda x: -x[1])[:10]
        for k, v in top_10:
            if k.startswith("f"):
                idx = int(k.replace("f", ""))
                if idx < len(data["feature_cols"]):
                    top_10_feats.append((data["feature_cols"][idx], float(v)))
    except:
        pass
    
    # Results
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] RESULTS")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Time: {train_time:.1f}s ({train_time/60:.1f}min)")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Test F1: {test_f1:.4f} | Prec: {test_prec:.4f} | Recall: {test_rec:.4f}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Per-Class Acc: SELL={per_class_acc.get('SELL',0):.4f} | HOLD={per_class_acc.get('HOLD',0):.4f} | BUY={per_class_acc.get('BUY',0):.4f}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]   Confusion Matrix:")
    print(f"[{datetime.now().strftime('%H:%M:%S')}]          SELL    HOLD     BUY")
    for i, rn in enumerate(["SELL", "HOLD", "BUY"]):
        print(f"[{datetime.now().strftime('%H:%M:%S')}]     {rn:5s} {cm[i][0]:6,d} {cm[i][1]:7,d} {cm[i][2]:7,d}")
    if top_10_feats:
        print(f"[{datetime.now().strftime('%H:%M:%S')}]   Top 5 Features:")
        for j, (fn, fv) in enumerate(top_10_feats[:5], 1):
            print(f"[{datetime.now().strftime('%H:%M:%S')}]     {j}. {fn[:45]:45s} {fv:6.0f}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
    
    # Cleanup
    del model, dtrain, dtest, y_train, y_test, y_pred_train, y_pred_test
    gc.collect()
    
    ram = psutil.virtual_memory()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] RAM: {ram.percent:.1f}% used ({ram.available/(1024**3):.1f}GB free)")
    
    return {
        "config": config,
        "train_time": train_time,
        "best_iteration": best_iter,
        "train_acc": train_acc,
        "test_acc": test_acc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_f1": test_f1,
        "per_class_acc": per_class_acc,
        "confusion_matrix": cm.tolist(),
        "top_10_features": top_10_feats,
    }

# ============================================================================
# TRAIN ALL MODELS
# ============================================================================

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] 🔥 TRAINING PIPELINE START")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")

pipeline_start = time.time()
results = []

for i, config in enumerate(MODEL_CONFIGS, 1):
    res = train_model(config, data, i, len(MODEL_CONFIGS))
    results.append(res)

pipeline_time = time.time() - pipeline_start

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ ALL MODELS TRAINED")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Total: {pipeline_time:.1f}s ({pipeline_time/60:.1f}min)")
print(f"[{datetime.now().strftime('%H:%M:%S')}]   Avg: {pipeline_time/len(results):.1f}s/model")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")

# ============================================================================
# RESULTS SUMMARY
# ============================================================================

summary = []
for res in results:
    summary.append({
        "name": res["config"]["name"],
        "style": res["config"]["style"],
        "horizon": res["config"]["horizon"],
        "train_acc": res["train_acc"],
        "test_acc": res["test_acc"],
        "test_f1": res["test_f1"],
        "test_precision": res["test_precision"],
        "test_recall": res["test_recall"],
        "train_time": res["train_time"],
    })

summary_sorted = sorted(summary, key=lambda x: x["test_f1"], reverse=True)

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] {'='*110}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] 🏆 MODEL RANKING (by Test F1)")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*110}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'#':<3} {'Model':<35} {'Style':<12} {'H':<8} {'Acc':<7} {'F1':<7} {'Prec':<7} {'Rec':<7} {'Time':<8}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'-'*110}")
for i, row in enumerate(summary_sorted, 1):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {i:<3} {row['name']:<35} {row['style']:<12} {str(row['horizon']):<8} "
          f"{row['test_acc']:<7.4f} {row['test_f1']:<7.4f} {row['test_precision']:<7.4f} "
          f"{row['test_recall']:<7.4f} {row['train_time']:<8.1f}s")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*110}")

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 🎯 BEST PER STYLE:")
for style in ["scalper", "day_trader", "swing_trader", "all"]:
    style_results = [r for r in results if r["config"]["style"] == style]
    if style_results:
        best = max(style_results, key=lambda x: x["test_f1"])
        print(f"[{datetime.now().strftime('%H:%M:%S')}]   {style.upper():15s}: {best['config']['name']:35s} F1={best['test_f1']:.4f}")

# ============================================================================
# VISUALIZATION
# ============================================================================

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 📊 Generating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

names = [s["name"].replace("XGB_", "") for s in summary_sorted]
f1s = [s["test_f1"] for s in summary_sorted]
colors = ['#FF6B6B' if s["style"] == 'scalper' else '#4ECDC4' if s["style"] == 'day_trader' 
          else '#45B7D1' if s["style"] == 'swing_trader' else '#FFA07A' for s in summary_sorted]

axes[0, 0].barh(names, f1s, color=colors)
axes[0, 0].set_xlabel('Test F1')
axes[0, 0].set_title('F1 Score by Model')
axes[0, 0].grid(axis='x', alpha=0.3)

train_accs = [s["train_acc"] for s in summary_sorted]
test_accs = [s["test_acc"] for s in summary_sorted]
x = np.arange(len(names))
w = 0.35
axes[0, 1].bar(x - w/2, train_accs, w, label='Train', alpha=0.8)
axes[0, 1].bar(x + w/2, test_accs, w, label='Test', alpha=0.8)
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Train vs Test Accuracy')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([n[:15] for n in names], rotation=45, ha='right', fontsize=8)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

times = [s["train_time"]/60 for s in summary_sorted]
axes[1, 0].barh(names, times, color='#FDCB6E')
axes[1, 0].set_xlabel('Time (min)')
axes[1, 0].set_title('Training Time')
axes[1, 0].grid(axis='x', alpha=0.3)

precs = [s["test_precision"] for s in summary_sorted]
recs = [s["test_recall"] for s in summary_sorted]
axes[1, 1].scatter(precs, recs, s=150, c=f1s, cmap='viridis', alpha=0.7, edgecolors='black')
axes[1, 1].set_xlabel('Precision')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_title('Precision vs Recall')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print(f"[{datetime.now().strftime('%H:%M:%S')}]   ✓ Saved: model_comparison.png")

# ============================================================================
# SAVE REPORT
# ============================================================================

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 💾 Saving report...")

report = {
    "metadata": {
        "timestamp": datetime.now().isoformat(),
        "dataset": DATASET_PATH,
        "train_samples": int(data["X_train"].shape[0]),
        "test_samples": int(data["X_test"].shape[0]),
        "num_features": int(data["X_train"].shape[1]),
        "total_time": pipeline_time,
    },
    "summary": [
        {
            "rank": i,
            "name": row["name"],
            "style": row["style"],
            "horizon": row["horizon"],
            "train_acc": row["train_acc"],
            "test_acc": row["test_acc"],
            "test_f1": row["test_f1"],
            "test_precision": row["test_precision"],
            "test_recall": row["test_recall"],
            "train_time": row["train_time"],
        }
        for i, row in enumerate(summary_sorted, 1)
    ],
    "best_per_style": {},
    "detailed_results": results,
}

for style in ["scalper", "day_trader", "swing_trader", "all"]:
    style_results = [r for r in results if r["config"]["style"] == style]
    if style_results:
        best = max(style_results, key=lambda x: x["test_f1"])
        report["best_per_style"][style] = {
            "name": best["config"]["name"],
            "test_f1": best["test_f1"],
            "test_acc": best["test_acc"],
        }

output_path = "training_report_gpu.json"
with open(output_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"[{datetime.now().strftime('%H:%M:%S')}]   ✓ Saved: {output_path}")

# ============================================================================
# DOWNLOAD FILES
# ============================================================================

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")
print(f"[{datetime.now().strftime('%H:%M:%S')}] 📥 DOWNLOADING RESULTS")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")

from google.colab import files
files.download(output_path)
files.download('model_comparison.png')

print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ COMPLETE!")
print(f"[{datetime.now().strftime('%H:%M:%S')}] {'='*80}")